# NOM / IR / PE — Domain Expansion: Trichotomous
3 agents, 4 items, ordinal preferences, ε(3)=0

In [ ]:
# ── Colab セットアップ（最初に1回だけ実行）──────────────────────────
!git clone https://github.com/shiro-seminar/NOM-matching.git

import sys
sys.path.insert(0, "/content/NOM-matching")

In [ ]:
# コードを更新したいときはここを実行
!git -C /content/NOM-matching pull

In [ ]:
import torch
from domain_expansion_experiments.config      import Config
from domain_expansion_experiments.domains     import DOMAINS
from domain_expansion_experiments.train       import train
from domain_expansion_experiments.evaluate    import evaluate_mechanism, print_table
from domain_expansion_experiments.benchmarks  import BENCHMARKS
from domain_expansion_experiments.data_gen    import sample_batch
from domain_expansion_experiments.model       import AllocationNet
from domain_expansion_experiments.allocations import ir_pe_mask

## 1. Config

In [ ]:
DOMAIN = "trichotomous"   # trichotomous / trichotomous_extended_e3 / four_chotomous_e4 / strict

cfg = Config(
    domain     = DOMAIN,
    steps      = 50_000,
    S          = 8,
    M          = 8,
    batch_size = 64,
    device     = "cuda" if torch.cuda.is_available() else "cpu",
    seed       = 42,
)
print(cfg)
print(f"device: {cfg.device}  num_ranks: {cfg.num_ranks}")

## 2. Train

In [ ]:
net = train(cfg, verbose=True)
net.eval()

## 3. Evaluate — benchmarks vs learned net

In [ ]:
EVAL_N  = 500    # evaluation sample size
EVAL_S  = 64     # NOM opponent samples
EVAL_M  = 64     # NOM misreport samples

eval_cfg = Config(domain=DOMAIN, batch_size=EVAL_N, device=cfg.device, seed=0)
torch.manual_seed(0)

domain = DOMAINS[DOMAIN]
batch  = sample_batch(eval_cfg)
marginal_rank = batch["marginal_rank"]
endow_idx     = batch["endow_idx"]
S             = batch["S"]

# WMAX: oracle upper bound on welfare (IR+PE constraint)
irpe_m = ir_pe_mask(eval_cfg, S, endow_idx)
wmax_s = (S.sum(1) + (1 - irpe_m) * (-1e9)).max(1).values

results = []
for bname, bfn in BENCHMARKS.items():
    print(f"Evaluating {bname}...")
    r = evaluate_mechanism(bname, bfn, eval_cfg, domain,
                           marginal_rank, endow_idx, S, wmax_s,
                           eval_S=EVAL_S, eval_M=EVAL_M)
    results.append(r)

def net_mech(cfg_, mr_, ei_, S_):
    mask_ = ir_pe_mask(cfg_, S_, ei_)
    return net(mr_, mask=mask_, temperature=1e-3)

results.append(evaluate_mechanism("LearnedNet", net_mech, eval_cfg, domain,
                                  marginal_rank, endow_idx, S, wmax_s,
                                  eval_S=EVAL_S, eval_M=EVAL_M))

print_table(results)

## 4. (Optional) チェックポイントを保存 / 読み込み

In [ ]:
# Google Drive にモデルを保存したい場合
from google.colab import drive
drive.mount("/content/drive")

import shutil
shutil.copy(f"{DOMAIN}_net.pt", f"/content/drive/MyDrive/{DOMAIN}_net.pt")
print(f"saved: {DOMAIN}_net.pt")

In [ ]:
# Drive から読み込む場合
CKPT = f"/content/drive/MyDrive/{DOMAIN}_net.pt"
ckpt = torch.load(CKPT, map_location=cfg.device)
net2 = AllocationNet(Config(**ckpt["cfg"]))
net2.load_state_dict(ckpt["state_dict"])
net2.eval()
print(f"loaded: step={ckpt.get('step', 'final')}")